In [1]:
# 데이터 다운로드
from roboflow import Roboflow

rf = Roboflow(api_key="Aov0WcVV5Y6GR71aEhV6")
project = rf.workspace("6rainstorm-yqytq").project("6rainstorm-final-project")
version = project.version(6)
dataset = version.download("yolov8", location="./solar_data")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to ./solar_data in yolov8:: 100%|██████████| 7292/7292 [00:04<00:00, 1762.97it/s]


In [2]:
# 클래스 매핑 
import yaml, os, shutil

CLASS_MAP = {
    0: 3, 1: 4, 2: 1,
    3: 0, 4: 4, 5: 2,
}
OUR_CLASSES = {
    0: 'normal', 1: 'dust', 2: 'snow',
    3: 'bird_dropping', 4: 'physical_damage',
}

src_base = './solar_data'
dst_base = './solar_data_mapped'

splits = {'train': 'train', 'valid': 'val', 'test': 'test'}

for src_split, dst_split in splits.items():
    src_label_dir = f'{src_base}/{src_split}/labels'
    src_image_dir = f'{src_base}/{src_split}/images'
    dst_label_dir = f'{dst_base}/{dst_split}/labels'
    dst_image_dir = f'{dst_base}/{dst_split}/images'

    os.makedirs(dst_label_dir, exist_ok=True)
    os.makedirs(dst_image_dir, exist_ok=True)

    for img in os.listdir(src_image_dir):
        shutil.copy(f'{src_image_dir}/{img}', f'{dst_image_dir}/{img}')

    for label_file in os.listdir(src_label_dir):
        src_path = f'{src_label_dir}/{label_file}'
        dst_path = f'{dst_label_dir}/{label_file}'
        with open(src_path, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if not parts:
                continue
            new_lines.append(f"{CLASS_MAP[int(parts[0])]} {' '.join(parts[1:])}\n")
        with open(dst_path, 'w') as f:
            f.writelines(new_lines)
    print(f"[{src_split} → {dst_split}] 변환 완료")

new_yaml = {
    'path': os.path.abspath('./solar_data_mapped'),  # 절대경로로 변환
    'train': 'train/images',
    'val':   'val/images',
    'test':  'test/images',
    'nc': 5,
    'names': {0:'normal', 1:'dust', 2:'snow',
              3:'bird_dropping', 4:'physical_damage'}
}
with open('./solar_data_mapped/dataset.yaml', 'w') as f:
    yaml.dump(new_yaml, f)
print("dataset.yaml 생성 완료!")

[train → train] 변환 완료
[valid → val] 변환 완료
[test → test] 변환 완료
dataset.yaml 생성 완료!


In [3]:
import os
print(os.path.exists('./solar_data_mapped'))
print(os.listdir('./solar_data_mapped') if os.path.exists('./solar_data_mapped') else "폴더 없음")

True
['dataset.yaml', 'test', 'train', 'val']


In [8]:
from ultralytics import RTDETR
import torch

print(f"GPU 사용 가능: {torch.cuda.is_available()}")
print(f"GPU 이름: {torch.cuda.get_device_name()}")

model = RTDETR('rtdetr-l.pt')

results = model.train(
    data='./solar_data_mapped/dataset.yaml',
    epochs=150,
    imgsz=640,
    batch=8,
    name='solar_panel_v4_rtdetr_l',
    patience=30,
    device=0,
    cls=2.0,
    hsv_h=0.015,
    hsv_s=0.7,
    hsv_v=0.4,
    flipud=0.3,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.15,
    copy_paste=0.3,
    save=True,
    save_period=10,
    project='./weights',
)

print(f"mAP50: {results.results_dict['metrics/mAP50(B)']:.4f}")

GPU 사용 가능: True
GPU 이름: NVIDIA GeForce RTX 3060 Ti
New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.10.20 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./solar_data_mapped/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=rtd

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\Context.cpp:95.)


      1/150      6.84G     0.8313      1.553     0.7098          7        640: 100% ━━━━━━━━━━━━ 363/363 2.9it/s 2:05<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 6.0it/s 4.3s0.2s
                   all        409       1560      0.632      0.368      0.362      0.253

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      2/150      6.68G     0.2672     0.8356     0.2341         34        640: 0% ──────────── 0/363  0.4s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\Context.cpp:95.)


      2/150      6.74G     0.6111     0.7903     0.4705         10        640: 100% ━━━━━━━━━━━━ 363/363 3.0it/s 1:60<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 6.1it/s 4.2s0.2s
                   all        409       1560        0.7      0.392      0.437        0.3

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      3/150      6.74G     0.8121      0.619     0.3157         89        640: 0% ──────────── 0/363  0.4s

...

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\Context.cpp:95.)


     49/150      6.59G     0.5081     0.6998     0.3859          9        640: 100% ━━━━━━━━━━━━ 363/363 1.5s/it 9:04<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 5.7it/s 4.5s0.2s
                   all        409       1560      0.537      0.572      0.555       0.38

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     50/150      6.73G     0.7634     0.5088     0.3756         63        640: 0% ──────────── 0/363  0.4s

grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\Context.cpp:95.)


     50/150      6.73G     0.5374     0.6863     0.3946         65        640: 19% ━━────────── 68/363 3.1it/s 22.7s<1:36


KeyboardInterrupt: 

In [1]:
from ultralytics import RTDETR
import torch

model = RTDETR(r'C:\Users\hp112\OneDrive\바탕 화면\minji\pannel\runs\detect\weights\solar_panel_v4_rtdetr_l\weights\last.pt')

results = model.train(resume=True)

New https://pypi.org/project/ultralytics/8.4.41 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.37  Python-3.10.20 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=2.0, cls_pw=0.0, compile=False, conf=None, copy_paste=0.3, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=solar_data_mapped\dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.3, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.15, mode=train, model=C:\Users\hp112\OneDrive\ \minji\pannel\runs\detect\weigh

c:\Users\hp112\anaconda3\envs\pannel\lib\site-packages\torch\autograd\graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\Context.cpp:95.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     74/150      6.98G     0.4735     0.6281     0.3487          7        640: 100% ━━━━━━━━━━━━ 363/363 2.9it/s 2:04<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 6.1it/s 4.3s0.2s
                   all        409       1560      0.604      0.505      0.505      0.359

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     75/150      6.79G     0.2179      0.476      0.212         34        640: 0% ──────────── 0/363  0.4s

c:\Users\hp112\anaconda3\envs\pannel\lib\site-packages\torch\autograd\graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\Context.cpp:95.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


     75/150       6.8G     0.4618     0.6291     0.3444         10        640: 100% ━━━━━━━━━━━━ 363/363 3.0it/s 2:00<0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 6.1it/s 4.2s0.2s
                   all        409       1560       0.64      0.548      0.566      0.401

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
     76/150      6.79G     0.5318     0.5396     0.1882         89        640: 0% ──────────── 0/363  0.4s

...

    145/150      6.74G     0.3062     0.4209     0.2454          6        640: 100% ━━━━━━━━━━━━ 363/363 3.1it/s 1:57<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 6.6it/s 3.9s0.2s
                   all        409       1560          0          0          0          0
WARNING Skipping checkpoint save at epoch 144: EMA contains NaN/Inf

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    146/150      6.64G     0.3679     0.5019     0.1658         55        640: 0% ──────────── 0/363  0.4s

c:\Users\hp112\anaconda3\envs\pannel\lib\site-packages\torch\autograd\graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\Context.cpp:95.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    146/150      6.73G     0.3053     0.4061     0.2446          4        640: 100% ━━━━━━━━━━━━ 363/363 3.1it/s 1:57<0.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 6.6it/s 3.9s0.2s
                   all        409       1560          0          0          0          0
WARNING Skipping checkpoint save at epoch 145: EMA contains NaN/Inf

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
    147/150      6.43G     0.6698     0.4916     0.3681         77        640: 0% ──────────── 0/363  0.3s

c:\Users\hp112\anaconda3\envs\pannel\lib\site-packages\torch\autograd\graph.py:825: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\Context.cpp:95.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


    147/150      6.52G     0.3019     0.4142     0.2407          2        640: 100% ━━━━━━━━━━━━ 363/363 3.0it/s 2:01<0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 6.4it/s 4.1s0.2s
                   all        409       1560          0          0          0          0
EarlyStopping: Training stopped early as no improvement observed in last 30 epochs. Best results observed at epoch 117, best model saved as best.pt.
To update EarlyStopping(patience=30) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.
WARNING Skipping checkpoint save at epoch 146: EMA contains NaN/Inf

74 epochs completed in 2.608 hours.
Optimizer stripped from C:\Users\hp112\OneDrive\ \minji\pannel\runs\detect\weights\solar_panel_v4_rtdetr_l\weights\last.pt, 66.3MB
Optimizer stripped from C:\Users\hp112\OneDrive\ \minji\pannel\runs\detect\weights\solar_panel_v4_rtdetr_l\weights\best.pt, 66.3MB

Validating

In [ ]:
from ultralytics import RTDETR
import time
import torch

model = RTDETR(r'C:\Users\hp112\OneDrive\바탕 화면\minji\pannel\runs\detect\weights\solar_panel_v4_rtdetr_l\weights\best.pt')

m = model.val(data='./solar_data_mapped/dataset.yaml', verbose=False)

# FPS 측정
dummy = torch.zeros(1, 3, 640, 640).to('cuda')
start = time.time()
for _ in range(100):
    model.predict(dummy, verbose=False)
fps = 100 / (time.time() - start)

class_names = ['normal', 'dust', 'snow', 'bird_dropping', 'physical_damage']

print(f"\n{'='*45}")
print(f"[RT-DETR-l]")
print(f"  mAP50:    {m.box.map50:.4f}")
print(f"  mAP50-95: {m.box.map:.4f}")
print(f"  Recall:   {m.box.r.mean():.4f}")
print(f"  FPS:      {fps:.1f}")
print(f"\n  클래스별 AP50 / Recall:")
for i, name in enumerate(class_names):
    ap = m.box.ap50[i]
    recall = m.box.r[i]
    status = "주의" if ap < 0.3 else "  보통" if ap < 0.5 else "양호"
    print(f"    {name:20s}: AP={ap:.3f} | Recall={recall:.3f} {status}")

Ultralytics 8.4.37  Python-3.10.20 torch-2.5.1 CUDA:0 (NVIDIA GeForce RTX 3060 Ti, 8192MiB)
rt-detr-l summary: 310 layers, 31,994,015 parameters, 0 gradients, 103.5 GFLOPs
val: Fast image access  (ping: 0.10.0 ms, read: 191.167.9 MB/s, size: 66.7 KB)
val: Scanning C:\Users\hp112\OneDrive\바탕 화면\minji\pannel\solar_data_mapped\val\labels.cache... 409 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 409/409  0.0s
WARNING Box and segment counts should be equal, but got len(segments) = 1304, len(boxes) = 1560. To resolve this only boxes will be used and all segments will be removed. To avoid this please supply either a detect or segment dataset, not a detect-segment mixed dataset.
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 26/26 3.3it/s 7.9s0.3s
                   all        409       1560      0.698      0.653      0.665      0.475
Speed: 1.6ms preprocess, 15.3ms inference, 0.0ms loss, 0.3ms postprocess per image
Results sa